# Clean Separate Component Smoothers Submission

Self-contained Kaggle submission notebook for exp115. It rebuilds the clean artifact ensemble, selects the row residual alpha, separate base/row per-well smoothers, and residual weights only from non-heldout OOF predictions, then writes `submission.csv`.

In [ ]:
from __future__ import annotations

import gc
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


MODEL_NAMES = [
    'catboost-1',
    'catboost-2',
    'catboost-3',
    'lightgbm-1',
    'lightgbm-2',
    'lightgbm-3',
    'lightgbm-4',
]


def _top_level_entries(path: Path) -> list[str]:
    if not path.exists():
        return []
    return [str(p) for p in sorted(path.iterdir())[:50]]


def _path_sample(paths: list[Path], limit: int = 25) -> list[str]:
    return [str(p) for p in paths[:limit]]


def _find_competition_root(input_root: Path) -> Path:
    candidates = [
        input_root / 'rogii-wellbore-geology-prediction',
        input_root / 'competitions' / 'rogii-wellbore-geology-prediction',
    ]
    for path in candidates:
        if (path / 'sample_submission.csv').exists():
            return path

    matches = sorted(input_root.rglob('sample_submission.csv'))
    if matches:
        return matches[0].parent

    raise FileNotFoundError(
        'Could not find competition input containing sample_submission.csv. '
        f'Top-level /kaggle/input entries: {_top_level_entries(input_root)}'
    )


def artifact_train_csv(root: Path) -> Path:
    nested = root / 'data' / 'train.csv'
    flat = root / 'train.csv'
    if nested.exists():
        return nested
    if flat.exists():
        return flat
    return nested


def artifact_oof_paths(root: Path) -> dict[str, Path]:
    paths: dict[str, Path] = {}
    for name in MODEL_NAMES:
        nested = root / 'models' / name / 'oof_preds.pkl'
        flat = root / f'{name}_oof_preds.pkl'
        if nested.exists():
            paths[name] = nested
        elif flat.exists():
            paths[name] = flat
    return paths


def _find_artifact_root(input_root: Path) -> Path:
    candidates = [
        input_root / 'wellbore-geology-prediction-artifacts',
        input_root / 'datasets' / 'wellbore-geology-prediction-artifacts',
        input_root / 'datasets' / 'ravaghi' / 'wellbore-geology-prediction-artifacts',
    ]
    for path in candidates:
        if artifact_train_csv(path).exists() and artifact_oof_paths(path):
            return path

    for train_csv in sorted(input_root.rglob('train.csv')):
        path = train_csv.parent.parent if train_csv.parent.name == 'data' else train_csv.parent
        if artifact_train_csv(path).exists() and artifact_oof_paths(path):
            return path

    train_hits = sorted(input_root.rglob('train.csv'))
    oof_hits = sorted(input_root.rglob('oof_preds.pkl')) + sorted(input_root.rglob('*_oof_preds.pkl'))
    raise FileNotFoundError(
        'Could not find artifact input containing data/train.csv plus model OOF pickle files. '
        f'Top-level /kaggle/input entries: {_top_level_entries(input_root)}; '
        f'train.csv hits: {_path_sample(train_hits)}; '
        f'oof hits: {_path_sample(oof_hits)}'
    )


def resolve_paths() -> tuple[Path, Path, Path]:
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.exists():
        return _find_competition_root(kaggle_input), _find_artifact_root(kaggle_input), Path('/kaggle/working')

    root = Path.cwd()
    if not (root / 'data').exists() and (root.parent / 'data').exists():
        root = root.parent

    comp_candidates = [
        root / 'data/raw/rogii-wellbore-geology-prediction',
        root / 'data/raw/competitions/rogii-wellbore-geology-prediction',
    ]
    art_candidates = [
        root / 'data/artifacts/wellbore-geology-prediction-artifacts',
        root / 'datasets/wellbore-geology-prediction-artifacts',
        root / 'datasets/ravaghi/wellbore-geology-prediction-artifacts',
    ]
    comp = next((p for p in comp_candidates if (p / 'sample_submission.csv').exists()), comp_candidates[0])
    art = next((p for p in art_candidates if artifact_train_csv(p).exists() and artifact_oof_paths(p)), art_candidates[0])
    return comp, art, Path.cwd()


COMP, ART, WORKING = resolve_paths()
print(f'[CFG] COMP={COMP} exists={COMP.exists()}')
print(f'[CFG] ART={ART} exists={ART.exists()}')
print(f'[CFG] WORKING={WORKING} exists={WORKING.exists()}')
ALPHAS = [0.001, 0.003, 0.01, 0.03, 0.1]
WELL_WEIGHTS = np.arange(0.4875, 0.5625 + 1e-12, 0.00625)
ROW_WEIGHTS = np.arange(-0.18, -0.115 + 1e-12, 0.005)
CHUNK_ROWS = 250_000
ROW_COLUMNS = [
    'last_known_tvt', 'pf_ancc', 'pf_ancc_std', 'pf_ancc_delta', 'pf_z', 'pf_z_delta', 'pf_vs_z',
    'beam_mean_d', 'beam_std_d', 'sc8_d', 'sc15_d', 'sc25_d', 'sc_cons_d', 'sc_ens_d', 'sc_trust',
    'hyb_d', 'sig_std', 'sig_mean_d', 'tw_range', 'tw_gr_mean', 'grm21', 'grs21', 'grm51', 'grs51',
    'grm101', 'grs101', 'glag1', 'glead1', 'glag5', 'glead5', 'glag15', 'glead15', 'tdsc-15',
    'tdsc-8', 'tdsc0', 'tdsc8', 'tdsc15', 'tdpf-15', 'tdpf-8', 'tdpf0', 'tdpf8', 'tdpf15',
]


def rmse(pred: np.ndarray, y: np.ndarray) -> float:
    err = pred.astype(np.float64) - y.astype(np.float64)
    return float(np.sqrt(np.mean(err * err)))


def well_equal_rmse(pred: np.ndarray, y: np.ndarray, wells: np.ndarray) -> float:
    frame = pd.DataFrame({
        'well': wells.astype(str),
        'sqerr': (pred.astype(np.float64) - y.astype(np.float64)) ** 2,
    })
    return float(np.sqrt(frame.groupby('well', sort=True)['sqerr'].mean().mean()))


def load_artifact_frame() -> tuple[pd.DataFrame, list[str]]:
    train_path = artifact_train_csv(ART)
    columns = pd.read_csv(train_path, nrows=0).columns.tolist()
    feature_cols = [c for c in columns if c not in ('well', 'id', 'target')]
    dtypes = {c: 'float32' for c in feature_cols + ['target']}
    dtypes.update({'well': 'string', 'id': 'string'})
    return pd.read_csv(train_path, dtype=dtypes), feature_cols


def load_artifact_members(last_known_tvt: np.ndarray) -> dict[str, np.ndarray]:
    paths = artifact_oof_paths(ART)
    missing = [name for name in MODEL_NAMES if name not in paths]
    if missing:
        raise FileNotFoundError(f'Missing OOF artifacts for {missing}; available={sorted(paths)}; ART={ART}')
    members: dict[str, np.ndarray] = {}
    for name in MODEL_NAMES:
        delta = np.asarray(joblib.load(paths[name]), dtype=np.float32)
        members[name] = (last_known_tvt + delta).astype(np.float32)
    return members


def gbr_d4() -> GradientBoostingRegressor:
    return GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=4,
        min_samples_leaf=5,
        random_state=42,
    )


def ridge(alpha: float):
    return make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha, solver='lsqr', fit_intercept=True),
    )


def build_well_meta(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    core_tokens = ('last_known_tvt', 'pf_', 'beam_', 'sc', 'hyb', 'sig_', 'tw_', 'gr', 'frm_rmse')
    core_cols = [c for c in feature_cols if any(token in c for token in core_tokens)]
    agg = {c: ['mean'] for c in feature_cols}
    for col in core_cols:
        agg[col].append('std')
    meta = df[['well'] + feature_cols].groupby('well', sort=True).agg(agg)
    meta.columns = ['__'.join(col).strip('_') for col in meta.columns.to_flat_index()]
    row_count = df.groupby('well', sort=True).size().rename('row_count').astype('float32')
    return meta.join(row_count).fillna(0.0)


def build_row_features(
    df: pd.DataFrame,
    wells: np.ndarray,
    base: np.ndarray,
    artifact_members: dict[str, np.ndarray],
) -> tuple[np.ndarray, list[str]]:
    selected_cols = [col for col in ROW_COLUMNS if col in df.columns]
    feature_blocks = [df[selected_cols].to_numpy(np.float32, copy=True)]
    feature_names = list(selected_cols)

    member_names = sorted(artifact_members)
    member_stack = np.vstack([artifact_members[name] for name in member_names]).astype(np.float32)
    artifact_std = member_stack.std(axis=0).astype(np.float32)
    artifact_range = (member_stack.max(axis=0) - member_stack.min(axis=0)).astype(np.float32)
    cat_names = [name for name in member_names if name.startswith('catboost')]
    lgb_names = [name for name in member_names if name.startswith('lightgbm')]
    cat_mean = np.mean([artifact_members[name] for name in cat_names], axis=0).astype(np.float32)
    lgb_mean = np.mean([artifact_members[name] for name in lgb_names], axis=0).astype(np.float32)

    row_num = df.groupby('well', sort=False).cumcount().to_numpy(np.float32)
    row_count = df.groupby('well', sort=False)['id'].transform('size').to_numpy(np.float32)
    row_frac = row_num / np.maximum(row_count - 1.0, 1.0)
    last = df['last_known_tvt'].to_numpy(np.float32)
    grouped = pd.DataFrame({'well': wells, 'base': base, 'last': last})
    base_centered = (grouped['base'] - grouped.groupby('well', sort=False)['base'].transform('mean')).to_numpy(np.float32)
    last_centered = (grouped['last'] - grouped.groupby('well', sort=False)['last'].transform('mean')).to_numpy(np.float32)

    derived = np.column_stack([
        base.astype(np.float32),
        artifact_std,
        artifact_range,
        cat_mean,
        lgb_mean,
        (cat_mean - lgb_mean).astype(np.float32),
        row_frac.astype(np.float32),
        np.log1p(row_count).astype(np.float32),
        base_centered,
        last_centered,
    ]).astype(np.float32)
    feature_blocks.append(derived)
    feature_names.extend([
        'artifact_base', 'artifact_std', 'artifact_range', 'catboost_mean', 'lightgbm_mean',
        'catboost_minus_lightgbm', 'row_frac', 'log_row_count', 'base_centered_by_well',
        'last_centered_by_well',
    ])
    X = np.column_stack(feature_blocks).astype(np.float32)
    del member_stack, derived, feature_blocks
    gc.collect()
    return X, feature_names


def make_well_oof(
    *,
    Xw: np.ndarray,
    well_target: np.ndarray,
    dev_well_positions: np.ndarray,
    folds: list[tuple[np.ndarray, np.ndarray]],
) -> np.ndarray:
    oof_well = np.zeros(dev_well_positions.shape[0], dtype=np.float32)
    for tr_rel, va_rel in folds:
        tr_pos = dev_well_positions[tr_rel]
        va_pos = dev_well_positions[va_rel]
        model = gbr_d4()
        model.fit(Xw[tr_pos], well_target[tr_pos])
        oof_well[va_rel] = model.predict(Xw[va_pos]).astype(np.float32)
        del model
        gc.collect()
    return oof_well


def smoother_candidates(prefix: str) -> list[dict[str, object]]:
    candidates: list[dict[str, object]] = [{'name': f'{prefix}_none', 'kind': 'none', 'base_name': 'none'}]
    for window in (501, 531, 551, 571, 601, 631):
        for shrink in (0.75, 0.825, 0.875, 0.925, 1.0):
            base_name = f'savgol_w{window}_p2_s{shrink:g}'
            candidates.append({
                'name': f'{prefix}_{base_name}',
                'base_name': base_name,
                'kind': 'savgol',
                'window': window,
                'polyorder': 2,
                'shrink': shrink,
            })
    return candidates


def _candidate_for_apply(candidate: dict[str, object]) -> dict[str, object]:
    if candidate['kind'] == 'none':
        return {'name': 'none', 'kind': 'none'}
    return {
        'name': candidate['base_name'],
        'kind': candidate['kind'],
        'window': candidate['window'],
        'polyorder': candidate['polyorder'],
        'shrink': candidate['shrink'],
    }


def build_smoothed_matrix(
    values: np.ndarray,
    wells: np.ndarray,
    candidates: list[dict[str, object]],
    *,
    label: str,
) -> np.ndarray:
    out = np.empty((values.shape[0], len(candidates)), dtype=np.float32)
    for col, candidate in enumerate(candidates):
        start = time.time()
        out[:, col] = apply_by_well(values, wells, _candidate_for_apply(candidate))
        print(f'{label} smoother {candidate["name"]} built in {time.time() - start:.1f}s')
    return out


def column_dots(matrix: np.ndarray, vector: np.ndarray) -> np.ndarray:
    out = np.zeros(matrix.shape[1], dtype=np.float64)
    for start in range(0, matrix.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, matrix.shape[0])
        out += matrix[start:stop].astype(np.float64).T @ vector[start:stop].astype(np.float64)
    return out


def column_squares(matrix: np.ndarray) -> np.ndarray:
    out = np.zeros(matrix.shape[1], dtype=np.float64)
    for start in range(0, matrix.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, matrix.shape[0])
        block = matrix[start:stop].astype(np.float64)
        out += np.sum(block * block, axis=0)
    return out


def cross_dots(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    out = np.zeros((left.shape[1], right.shape[1]), dtype=np.float64)
    for start in range(0, left.shape[0], CHUNK_ROWS):
        stop = min(start + CHUNK_ROWS, left.shape[0])
        out += left[start:stop].astype(np.float64).T @ right[start:stop].astype(np.float64)
    return out


def best_weight_grid(
    *,
    rr: float,
    r_well: float,
    r_row: float,
    well_well: float,
    well_row: float,
    row_row: float,
    n_rows: int,
) -> dict[str, float]:
    best = None
    n = float(n_rows)
    for well_weight in WELL_WEIGHTS:
        ww = float(well_weight)
        for row_weight in ROW_WEIGHTS:
            rw = float(row_weight)
            mse = (
                rr
                - 2.0 * ww * r_well
                - 2.0 * rw * r_row
                + ww * ww * well_well
                + 2.0 * ww * rw * well_row
                + rw * rw * row_row
            ) / n
            score = float(np.sqrt(max(mse, 0.0)))
            if best is None or score < best['dev_oof_rmse']:
                best = {'well_weight': ww, 'row_weight': rw, 'dev_oof_rmse': score}
    return best


def evaluate_component_grid(
    *,
    y: np.ndarray,
    base_matrix: np.ndarray,
    row_matrix: np.ndarray,
    well_component: np.ndarray,
    base_candidates: list[dict[str, object]],
    row_candidates: list[dict[str, object]],
) -> list[dict[str, object]]:
    y64 = y.astype(np.float64)
    well64 = well_component.astype(np.float64)
    yy = float(np.dot(y64, y64))
    y_well = float(np.dot(y64, well64))
    well_well = float(np.dot(well64, well64))
    y_base = column_dots(base_matrix, y)
    base_base = column_squares(base_matrix)
    base_well = column_dots(base_matrix, well_component)
    y_row = column_dots(row_matrix, y)
    row_row = column_squares(row_matrix)
    row_well = column_dots(row_matrix, well_component)
    base_row = cross_dots(base_matrix, row_matrix)

    rows = []
    n_rows = int(y.shape[0])
    for base_idx, base_candidate in enumerate(base_candidates):
        rr = yy - 2.0 * y_base[base_idx] + base_base[base_idx]
        r_well = y_well - base_well[base_idx]
        for row_idx, row_candidate in enumerate(row_candidates):
            r_row = y_row[row_idx] - base_row[base_idx, row_idx]
            combo = best_weight_grid(
                rr=rr,
                r_well=r_well,
                r_row=r_row,
                well_well=well_well,
                well_row=row_well[row_idx],
                row_row=row_row[row_idx],
                n_rows=n_rows,
            )
            rows.append({
                'base_smoother': _candidate_for_apply(base_candidate),
                'base_smoother_name': str(base_candidate['base_name']),
                'row_smoother': _candidate_for_apply(row_candidate),
                'row_smoother_name': str(row_candidate['base_name']),
                'well_weight': combo['well_weight'],
                'row_weight': combo['row_weight'],
                'dev_oof_rmse': combo['dev_oof_rmse'],
            })
    rows.sort(key=lambda row: float(row['dev_oof_rmse']))
    return rows


def add_selected_diagnostics(
    rows: list[dict[str, object]],
    *,
    y: np.ndarray,
    wells: np.ndarray,
    base_matrix: np.ndarray,
    row_matrix: np.ndarray,
    well_component: np.ndarray,
    base_candidates: list[dict[str, object]],
    row_candidates: list[dict[str, object]],
) -> list[dict[str, object]]:
    base_pos = {str(candidate['base_name']): pos for pos, candidate in enumerate(base_candidates)}
    row_pos = {str(candidate['base_name']): pos for pos, candidate in enumerate(row_candidates)}
    enriched = []
    for row in rows:
        base_idx = base_pos[str(row['base_smoother_name'])]
        row_idx = row_pos[str(row['row_smoother_name'])]
        pred = (
            base_matrix[:, base_idx]
            + float(row['well_weight']) * well_component
            + float(row['row_weight']) * row_matrix[:, row_idx]
        ).astype(np.float32)
        enriched.append(row | {
            'dev_well_equal_rmse': well_equal_rmse(pred, y, wells),
            'dev_bias': float(np.mean(pred.astype(np.float64) - y.astype(np.float64))),
        })
    return enriched


def _odd_window(requested: int, n: int, polyorder: int = 0) -> int | None:
    if n <= polyorder + 2:
        return None
    window = min(int(requested), n if n % 2 else n - 1)
    if window <= polyorder:
        window = polyorder + 2
        if window % 2 == 0:
            window += 1
    if window > n:
        window = n if n % 2 else n - 1
    if window <= polyorder or window < 3:
        return None
    return window


def smooth_values(values: np.ndarray, candidate: dict[str, object]) -> np.ndarray:
    if str(candidate['kind']) == 'none':
        return values.astype(np.float32, copy=True)
    if str(candidate['kind']) != 'savgol':
        raise ValueError(f"unsupported postprocess kind: {candidate['kind']}")
    polyorder = int(candidate['polyorder'])
    window = _odd_window(int(candidate['window']), len(values), polyorder)
    if window is None:
        return values.astype(np.float32, copy=True)
    smoothed = values.astype(np.float64, copy=True)
    target = savgol_filter(smoothed, window_length=window, polyorder=polyorder, mode='interp')
    shrink = float(candidate['shrink'])
    return (smoothed + shrink * (target - smoothed)).astype(np.float32)


def apply_by_well(pred: np.ndarray, wells: np.ndarray, candidate: dict[str, object]) -> np.ndarray:
    out = pred.astype(np.float32, copy=True)
    frame = pd.DataFrame({'well': wells.astype(str), 'pos': np.arange(len(wells), dtype=np.int64)})
    for _, positions in frame.groupby('well', sort=False)['pos']:
        idx = positions.to_numpy(dtype=np.int64)
        out[idx] = smooth_values(out[idx], candidate)
    return out


t0 = time.time()
df, feature_cols = load_artifact_frame()
wells = df['well'].astype(str).to_numpy()
ids = df['id'].astype(str).to_numpy()
last = df['last_known_tvt'].to_numpy(np.float32)
y = (last + df['target'].to_numpy(np.float32)).astype(np.float32)

sample_ids = pd.read_csv(COMP / 'sample_submission.csv', usecols=['id'], dtype={'id': 'string'})
sample_id_values = sample_ids['id'].astype(str).to_numpy()
sample_row_mask = np.isin(ids, sample_id_values)
if int(sample_row_mask.sum()) != len(sample_ids):
    raise ValueError('sample_submission ids do not align to artifact train.csv ids')
heldout_wells = np.unique(wells[sample_row_mask])
heldout = np.isin(wells, heldout_wells)
dev = ~heldout
idx_dev = np.flatnonzero(dev)

artifact_members = load_artifact_members(last)
base = np.mean(list(artifact_members.values()), axis=0).astype(np.float32)
residual = (y - base).astype(np.float32)

meta = build_well_meta(df, feature_cols)
well_ids = meta.index.astype(str).to_numpy()
Xw = meta.to_numpy(np.float32)
well_target = (
    pd.DataFrame({'well': wells, 'residual': residual})
    .groupby('well', sort=True)['residual']
    .mean()
    .loc[well_ids]
    .to_numpy(np.float32)
)
well_to_pos = {well: pos for pos, well in enumerate(well_ids)}
row_well_pos = np.array([well_to_pos[well] for well in wells], dtype=np.int32)
held_well_mask = np.isin(well_ids, heldout_wells)
dev_well_positions = np.flatnonzero(~held_well_mask)
held_well_positions = np.flatnonzero(held_well_mask)
well_folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(dev_well_positions))
well_oof = make_well_oof(Xw=Xw, well_target=well_target, dev_well_positions=dev_well_positions, folds=well_folds)
correction_by_well = np.zeros(len(well_ids), dtype=np.float32)
correction_by_well[dev_well_positions] = well_oof
well_row_oof = correction_by_well[row_well_pos]

X, row_feature_names = build_row_features(df, wells, base, artifact_members)
del artifact_members
gc.collect()

group_folds = list(GroupKFold(n_splits=3).split(np.zeros(idx_dev.shape[0]), groups=wells[dev]))
y_dev = y[idx_dev]
wells_dev = wells[idx_dev]
well_dev = well_row_oof[idx_dev].astype(np.float32)
base_candidates = smoother_candidates('base')
row_candidates = smoother_candidates('row')
base_matrix = build_smoothed_matrix(base[idx_dev], wells_dev, base_candidates, label='base')
alpha_results = []
best_selection = None
for alpha in ALPHAS:
    row_oof = np.zeros(idx_dev.shape[0], dtype=np.float32)
    fold_scores = []
    for fold_idx, (tr_rel, va_rel) in enumerate(group_folds):
        tr_idx = idx_dev[tr_rel]
        va_idx = idx_dev[va_rel]
        model = ridge(alpha)
        model.fit(X[tr_idx], residual[tr_idx])
        pred = model.predict(X[va_idx]).astype(np.float32)
        row_oof[va_rel] = pred
        fold_scores.append({
            'fold': fold_idx,
            'raw_row_rmse': rmse(base[va_idx] + pred, y[va_idx]),
            'rows': int(va_idx.size),
            'wells': int(np.unique(wells[va_idx]).size),
        })
        print(f'alpha={alpha:g} fold={fold_idx} raw_row={fold_scores[-1]["raw_row_rmse"]:.4f}')
        del model
        gc.collect()

    row_matrix = build_smoothed_matrix(row_oof, wells_dev, row_candidates, label=f'row alpha={alpha:g}')
    pair_rows = evaluate_component_grid(
        y=y_dev,
        base_matrix=base_matrix,
        row_matrix=row_matrix,
        well_component=well_dev,
        base_candidates=base_candidates,
        row_candidates=row_candidates,
    )
    top_rows = add_selected_diagnostics(
        pair_rows[:25],
        y=y_dev,
        wells=wells_dev,
        base_matrix=base_matrix,
        row_matrix=row_matrix,
        well_component=well_dev,
        base_candidates=base_candidates,
        row_candidates=row_candidates,
    )
    alpha_best = sorted(top_rows, key=lambda row: (row['dev_oof_rmse'], row['dev_well_equal_rmse']))[0] | {'alpha': float(alpha)}
    alpha_results.append({
        'alpha': float(alpha),
        'fold_scores': fold_scores,
        'best': alpha_best,
        'top_pairs': top_rows,
    })
    print(
        'alpha={alpha:g} best rmse={rmse:.6f} base={base_name} row={row_name} ww={ww:.4f} rw={rw:.4f}'.format(
            alpha=alpha,
            rmse=float(alpha_best['dev_oof_rmse']),
            base_name=alpha_best['base_smoother_name'],
            row_name=alpha_best['row_smoother_name'],
            ww=float(alpha_best['well_weight']),
            rw=float(alpha_best['row_weight']),
        )
    )
    if best_selection is None or (
        alpha_best['dev_oof_rmse'],
        alpha_best['dev_well_equal_rmse'],
    ) < (
        best_selection['dev_oof_rmse'],
        best_selection['dev_well_equal_rmse'],
    ):
        best_selection = dict(alpha_best)
    del row_matrix, row_oof
    gc.collect()

final_well = gbr_d4()
final_well.fit(Xw[dev_well_positions], well_target[dev_well_positions])
held_well_correction = final_well.predict(Xw[held_well_positions]).astype(np.float32)
del final_well
gc.collect()

if abs(best_selection['row_weight']) > 1e-12:
    final_row = ridge(best_selection['alpha'])
    final_row.fit(X[dev], residual[dev])
    held_row_residual = final_row.predict(X[heldout]).astype(np.float32)
    del final_row
else:
    held_row_residual = np.zeros(int(heldout.sum()), dtype=np.float32)
del X
gc.collect()

correction_by_well = np.zeros(len(well_ids), dtype=np.float32)
correction_by_well[held_well_positions] = held_well_correction
held_well_row = correction_by_well[row_well_pos][heldout]
base_smoother = dict(best_selection['base_smoother'])
row_smoother = dict(best_selection['row_smoother'])
smooth_base = apply_by_well(base[heldout], wells[heldout], base_smoother)
smooth_row = apply_by_well(held_row_residual, wells[heldout], row_smoother)
raw_clean_pred = (
    base[heldout]
    + float(best_selection['well_weight']) * held_well_row
    + float(best_selection['row_weight']) * held_row_residual
).astype(np.float32)
clean_pred = (
    smooth_base
    + float(best_selection['well_weight']) * held_well_row
    + float(best_selection['row_weight']) * smooth_row
).astype(np.float32)

clean_rows = pd.DataFrame({'id': ids[heldout], 'tvt': clean_pred.astype(np.float32)})
submission = sample_ids.astype({'id': 'string'}).merge(clean_rows, on='id', how='left')
if int(submission['tvt'].isna().sum()):
    raise ValueError('missing submission predictions')
submission.to_csv(WORKING / 'submission.csv', index=False)

print('heldout_wells:', heldout_wells.tolist())
print('selected:', best_selection)
print('selected_base_smoother:', base_smoother)
print('selected_row_smoother:', row_smoother)
print('row_feature_count:', len(row_feature_names))
print('local_raw_holdout_rmse:', rmse(raw_clean_pred, y[heldout]))
print('local_clean_holdout_rmse:', rmse(clean_pred, y[heldout]))
print('local_clean_holdout_mse:', rmse(clean_pred, y[heldout]) ** 2)
print('wrote:', WORKING / 'submission.csv')
print('elapsed_seconds:', time.time() - t0)